# Validation and preservation / 验证与保留

Unknown XML content and untouched source lexemes survive unrelated typed edits. Validation reports known schema issues without rewriting input.

无关类型化修改会保留未知 XML 内容和未修改的源词法形式。验证器报告已知 schema 问题，不会改写输入。

In [1]:
from cdxml_om import Bond, CDXMLDocument, Node, UnknownElement

source = (
    '<CDXML xmlns:v="urn:vendor"><page id="1"><fragment id="2">'
    '<n id="3" Element="6" p="001.000 02.500" Charge="0" VendorNode="keep"/>'
    '<!-- retain sibling position --><v:opaque marker="preserve">opaque payload</v:opaque>'
    '<n id="4" Element="8" p="3 4"/></fragment></page></CDXML>'
)
document = CDXMLDocument.from_string(source)
fragment = document.pages[0].fragments[0]
first_node = document.get(Node, 3)
opaque = next(child for child in fragment.children if isinstance(child, UnknownElement))
assert first_node is not None and first_node.position is not None
assert opaque.raw_attributes["marker"] == "preserve"
first_node.charge = 1
serialized = document.to_string()
assert 'VendorNode="keep"' in serialized
assert 'p="001.000 02.500"' in serialized
assert "<!-- retain sibling position -->" in serialized
assert 'marker="preserve"' in serialized and "opaque payload" in serialized
assert serialized.index('<n id="3"') < serialized.index("<v:opaque") < serialized.index('<n id="4"')
before_validation = document.to_string()
assert document.validate().is_valid
assert document.to_string() == before_validation
reloaded = CDXMLDocument.from_string(serialized)
reloaded_fragment = reloaded.pages[0].fragments[0]
reloaded_node = reloaded.get(Node, 3)
reloaded_opaque = next(
    child for child in reloaded_fragment.children if isinstance(child, UnknownElement)
)
assert reloaded_node is not None and reloaded_node.charge == 1
assert reloaded_node.raw_attributes["p"] == "001.000 02.500"
assert reloaded_node.raw_attributes["VendorNode"] == "keep"
assert reloaded_opaque.text == "opaque payload"
assert reloaded_opaque.raw_attributes["marker"] == "preserve"
roundtrip_xml = reloaded.to_string()
assert "<!-- retain sibling position -->" in roundtrip_xml
assert (
    roundtrip_xml.index('<n id="3"')
    < roundtrip_xml.index("<v:opaque")
    < roundtrip_xml.index('<n id="4"')
)
assert reloaded.validate().is_valid
print("Untouched XML and source lexemes survive parse/serialize/reload.")

Untouched XML and source lexemes survive parse/serialize/reload.


## Structured validation / 结构化验证

These separate inputs intentionally include duplicate IDs, a dangling reference, a wrong reference target, an invalid integer value, and an illegal parent.

以下独立输入分别故意包含重复 ID、悬空引用、错误引用目标、无效整数值及非法父元素。

In [2]:
def finding(xml, expected_code):
    document = CDXMLDocument.from_string(xml)
    before_validation = document.to_string()
    report = document.validate()
    assert not report.is_valid
    assert document.to_string() == before_validation
    issue = next(issue for issue in report.errors if issue.code == expected_code)
    return issue.code, issue.property_id or "document structure"


duplicate = finding(
    '<CDXML><page id="1"><fragment id="2"><n id="3" Element="6"/>'
    '<n id="3" Element="8"/></fragment></page></CDXML>',
    "duplicate-id",
)
dangling_xml = (
    '<CDXML><page id="1"><fragment id="2"><n id="3" Element="6"/>'
    '<n id="4" Element="8"/><b id="5" B="3" E="999"/>'
    "</fragment></page></CDXML>"
)
dangling = finding(dangling_xml, "dangling-reference")
dangling_document = CDXMLDocument.from_string(dangling_xml)
assert dangling_document.find(Bond)[0].raw_reference_id("bond.end") == 999
dangling_reloaded = CDXMLDocument.from_string(dangling_document.to_string())
assert dangling_reloaded.find(Bond)[0].raw_reference_id("bond.end") == 999
wrong_target = finding(
    '<CDXML><page id="1"><fragment id="2"><n id="3" Element="6"/>'
    '<b id="5" B="4" E="3"/></fragment>'
    '<fragment id="4"><n id="6" Element="8"/></fragment></page></CDXML>',
    "wrong-reference-target",
)
bad_value = finding(
    '<CDXML><page id="1"><fragment id="2"><n id="3" Element="6" Charge="NaN"/>'
    "</fragment></page></CDXML>",
    "invalid-value",
)
bad_parent = finding('<CDXML><n id="3" Element="6"/></CDXML>', "invalid-parent")
assert duplicate == ("duplicate-id", "common.id")
assert dangling[0] == "dangling-reference"
assert wrong_target == ("wrong-reference-target", "bond.begin")
assert bad_value == ("invalid-value", "node.charge")
assert bad_parent[0] == "invalid-parent"
diagnostics = (duplicate, dangling, wrong_target, bad_value, bad_parent)
print("Structured findings: " + "; ".join(f"{code} ({field})" for code, field in diagnostics))

Structured findings: duplicate-id (common.id); dangling-reference (bond.end); wrong-reference-target (bond.begin); invalid-value (node.charge); invalid-parent (document structure)


## External entities stay opaque / 外部实体保持不透明

The parser disables external-entity loading and network access. A harmless temporary sentinel demonstrates one unresolved external entity case; it is not a complete grammar or chemistry check. See the [feature scope](../../docs/feature-support.md).

解析器禁用外部实体加载和网络访问。下面用无害的临时哨兵文件演示外部实体引用保持未解析状态。

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix="cdxml-om-entity-example-") as temporary_directory:
    sentinel = Path(temporary_directory) / "sentinel.txt"
    sentinel.write_text("MUST_NOT_BE_READ_OR_SERIALIZED", encoding="utf-8")
    source = (
        f'<!DOCTYPE CDXML [<!ENTITY local SYSTEM "{sentinel.as_uri()}">]>'
        '<CDXML><page id="1"><fragment id="2">'
        '<n id="3" Element="6"/><v:opaque xmlns:v="urn:vendor">&local;</v:opaque>'
        "</fragment></page></CDXML>"
    )
    secured = CDXMLDocument.from_string(source)
    entity_xml = secured.to_string()
    assert "&local;" in entity_xml
    assert "MUST_NOT_BE_READ_OR_SERIALIZED" not in entity_xml
    entity = secured.tree.xpath("//v:opaque", namespaces={"v": "urn:vendor"})[0][0]
    assert entity.name == "local"

print("External entity content was not expanded into the XML tree or serialization.")
print("CDXML-OM notebook: 03_validation_and_preservation OK")

External entity content was not expanded into the XML tree or serialization.
CDXML-OM notebook: 03_validation_and_preservation OK
